# 11 · Отбор с отложенным типом требований (val + LOTO)

**Зачем** (JOURNAL, решение 2026-10-31). На val те же типы требований, что в train, поэтому отбор по val не видит
переноса: в 09 по val `linmap` (0.86) обходил `linmap_prompt` (0.74), а на новых типах — наоборот (0.64 против 0.83).

**Отбор val+loto.** Для каждого из типов deadline / quantity / budget карта подгоняется на train **без** задач
этого типа и проверяется на них — как будто тип новый. Выбирается конфигурация с максимумом среднего успеха
на val и на отложенном типе (специфичность на val — не ниже допуска). Выбор по одному val оценивается рядом
как `m@val` — так видно, что дал новый отбор.

Методы без градиента: `linmap`, `linmap_prompt`, `linmap_mix`; ориентир — LoRA. Проверка — на **свежем**
наборе переноса (seed 505). Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [2]:
cfg = ExperimentConfig.make(
    name="loto-select-qwen3-1.7b", hypothesis="explore", notebook="notebooks/11_loto_select.ipynb",
    notes="Отбор val+loto (отложенный тип требований внутри train) для linmap, linmap_prompt, linmap_mix.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.select": "val+loto",
        "act.ranks": [8],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 505,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[16:09:44] run 20261031-160942_loto-select-qwen3-1.7b_bd79ce  ->  results/runs/20261031-160942_loto-select-qwen3-1.7b_bd79ce
[16:09:44] data: {'train': 79, 'val': 31, 'test': 50}  types={'invariant': 41, 'changing': 112, 'stress': 7}  positions={'early': 114, 'late': 46}


baseline/train:   0%|          | 0/158 [00:00<?, ?it/s]

[16:09:44] loading model Qwen/Qwen3-1.7B …


W1031 16:09:47.362000 17149 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (8,816 outputs left out)

linmap_prompt@val/val:   0%|          | 0/62 [00:00<?, ?it/s]

linmap_mix@val/val:   0%|          | 0/62 [00:00<?, ?it/s]

[16:56:17] val: baseline 0.621; lora 0.948; linmap 0.862; linmap_prompt 0.724; linmap_mix 0.793; linmap_prompt@val 0.776; linmap_mix@val 0.810


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.621,0.621,0.526,0.4,NaN,,NaN,NaN,None,None,,0.5,831,10.0,400.6
1,lora,0.948,0.948,1.000,0.8,0.328,"[+0.190, +0.466]",19.0,0.0,None,None,,0.0,831,10.0,299.0
2,linmap,0.862,0.862,0.842,0.6,0.241,"[+0.121, +0.379]",16.0,2.0,None,None,,0.5,831,10.0,410.6
3,linmap_prompt,0.724,0.724,0.579,0.6,0.103,"[+0.000, +0.224]",8.0,2.0,None,None,,0.5,831,10.0,298.0
4,linmap_mix,0.793,0.793,0.737,0.6,0.172,"[+0.069, +0.293]",11.0,1.0,None,None,,0.5,831,10.0,413.1
5,linmap_prompt@val,0.776,0.776,0.684,0.6,0.155,"[+0.052, +0.276]",11.0,2.0,None,None,,0.5,831,10.0,410.9
6,linmap_mix@val,0.810,0.810,0.789,0.6,0.190,"[+0.086, +0.310]",11.0,0.0,None,None,,0.5,831,10.0,421.6


### Подбор: val и отложенный тип

In [3]:
f = exp.state["fitted"]
cols = ["layer", "rank", "alpha", "success", "loto_success", "score", "specificity", "loto_specificity", "spec_drop"]
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    r, rv = f[m]["row"], f[m]["row_val"]
    display(Markdown(f"**{m}** — выбор val+loto: L{r['layer']}/α{r['alpha']} (score {r['score']:.3f}); "
                     f"по одному val: L{rv['layer']}/α{rv['alpha']} (score {rv['score']:.3f})"))
    display(t.sort_values("score", ascending=False)[cols].head(6).round(3))

**linmap** — выбор val+loto: L18/α1.5 (score 0.790); по одному val: L18/α1.5 (score 0.790)

,layer,rank,alpha,success,loto_success,score,specificity,loto_specificity,spec_drop
6,18,8,1.5,0.862,0.717,0.790,0.6,0.667,-0.2
10,20,8,1.0,0.793,0.707,0.750,0.5,0.722,-0.1
11,20,8,1.5,0.793,0.707,0.750,0.5,0.667,-0.1
1,16,8,1.5,0.810,0.685,0.748,0.6,0.611,-0.2
5,18,8,1.0,0.793,0.696,0.744,0.5,0.667,-0.1
3,16,8,3.0,0.759,0.717,0.738,0.6,0.667,-0.2


**linmap_prompt** — выбор val+loto: L18/α2.0 (score 0.721); по одному val: L24/α4.0 (score 0.681)

,layer,rank,alpha,success,loto_success,score,specificity,loto_specificity,spec_drop
7,18,8,2.0,0.724,0.717,0.721,0.6,0.722,-0.2
13,20,8,3.0,0.741,0.674,0.708,0.6,0.611,-0.2
10,20,8,1.0,0.707,0.696,0.701,0.5,0.722,-0.1
6,18,8,1.5,0.707,0.696,0.701,0.5,0.722,-0.1
11,20,8,1.5,0.707,0.685,0.696,0.5,0.722,-0.1
5,18,8,1.0,0.707,0.685,0.696,0.5,0.722,-0.1


**linmap_mix** — выбор val+loto: L18/α1.5 (score 0.782); по одному val: L18/α1.0 (score 0.780)

,layer,rank,alpha,success,loto_success,score,specificity,loto_specificity,spec_drop
6,18,8,1.5,0.793,0.772,0.782,0.6,0.778,-0.2
5,18,8,1.0,0.810,0.750,0.780,0.6,0.722,-0.2
7,18,8,2.0,0.776,0.783,0.779,0.6,0.778,-0.2
12,20,8,2.0,0.810,0.739,0.775,0.6,0.778,-0.2
11,20,8,1.5,0.793,0.750,0.772,0.6,0.778,-0.2
10,20,8,1.0,0.741,0.783,0.762,0.5,0.778,-0.1


In [4]:
# Насколько val и отложенный тип согласны между собой по всем конфигурациям
rows = []
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    t = exp.fit_table(m)
    if not t.empty:
        rows.append({"method": m, "n_configs": len(t),
                     "corr(val, loto)": t["success"].corr(t["loto_success"], method="spearman"),
                     "val_best": t["success"].max(), "loto_best": t["loto_success"].max()})
pd.DataFrame(rows).round(3)

,method,n_configs,"corr(val, loto)",val_best,loto_best
0,linmap,30,0.333,0.862,0.717
1,linmap_prompt,30,-0.584,0.776,0.728
2,linmap_mix,30,0.045,0.810,0.783


### Свежий набор переноса (seed 505): выбор val+loto против выбора по одному val

In [5]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:   1%|          | 1/100 [00:00<00:35,  2.75it/s]

baseline/transfer:seen:   2%|▏         | 2/100 [00:00<00:35,  2.77it/s]

baseline/transfer:seen:   3%|▎         | 3/100 [00:01<00:35,  2.76it/s]

baseline/transfer:seen:   4%|▍         | 4/100 [00:01<00:34,  2.75it/s]

baseline/transfer:seen:   5%|▌         | 5/100 [00:02<00:42,  2.25it/s]

... (2,423 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.51,0.51,0.273,0.294,NaN,,NaN,NaN,0.46,NaN,,None,886,10.0,470.6
1,lora,0.94,0.94,0.939,0.882,0.43,"[+0.320, +0.540]",44.0,1.0,0.73,0.27,"[+0.160, +0.380]",None,886,10.0,496.9
2,linmap,0.75,0.75,0.636,0.529,0.24,"[+0.120, +0.360]",30.0,6.0,0.56,0.10,"[-0.040, +0.230]",None,886,10.0,546.4
3,linmap_prompt,0.73,0.73,0.545,0.588,0.22,"[+0.130, +0.320]",25.0,3.0,0.60,0.14,"[+0.070, +0.220]",None,886,10.0,624.7
4,linmap_mix,0.84,0.84,0.788,0.706,0.33,"[+0.220, +0.440]",35.0,2.0,0.68,0.22,"[+0.120, +0.320]",None,886,10.0,603.4
5,linmap_prompt@val,0.70,0.70,0.455,0.588,0.19,"[+0.070, +0.310]",27.0,8.0,NaN,NaN,,None,886,10.0,641.4
6,linmap_mix@val,0.71,0.71,0.515,0.588,0.20,"[+0.120, +0.290]",20.0,0.0,NaN,NaN,,None,886,10.0,696.2


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.45,0.45,0.111,0.286,NaN,,NaN,NaN,0.33,NaN,,None,853,10.0,401.7
1,lora,0.89,0.89,0.889,0.714,0.44,"[+0.340, +0.540]",45.0,1.0,0.84,0.51,"[+0.390, +0.620]",None,853,10.0,418.1
2,linmap,0.62,0.62,0.444,0.357,0.17,"[+0.100, +0.240]",17.0,0.0,0.40,0.07,"[-0.020, +0.160]",None,853,10.0,408.3
3,linmap_prompt,0.78,0.78,0.750,0.429,0.33,"[+0.240, +0.420]",33.0,0.0,0.70,0.37,"[+0.270, +0.470]",None,853,10.0,409.1
4,linmap_mix,0.76,0.76,0.694,0.429,0.31,"[+0.230, +0.400]",31.0,0.0,0.65,0.32,"[+0.230, +0.410]",None,853,10.0,408.5
5,linmap_prompt@val,0.78,0.78,0.750,0.500,0.33,"[+0.240, +0.420]",34.0,1.0,NaN,NaN,,None,853,10.0,408.5
6,linmap_mix@val,0.68,0.68,0.556,0.357,0.23,"[+0.160, +0.310]",23.0,0.0,NaN,NaN,,None,853,10.0,408.4


In [6]:
rows = []
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    for g in ("seen", "unseen"):
        s = exp.state["summaries"][f"transfer:{g}"]
        a, b = s.get(m), s.get(f"{m}@val", s.get(m))
        if a:
            rows.append({"method": m, "group": g, "val+loto": a["success"], "val only": b["success"],
                         "same_config": f"{m}@val" not in s})
pd.DataFrame(rows)

,method,group,val+loto,val only,same_config
0,linmap,seen,0.75,0.75,True
1,linmap,unseen,0.62,0.62,True
2,linmap_prompt,seen,0.73,0.70,False
3,linmap_prompt,unseen,0.78,0.78,False
4,linmap_mix,seen,0.84,0.71,False
5,linmap_mix,unseen,0.76,0.68,False


## Вывод → журнал

In [7]:
display(Markdown(exp.report(
    conclusion="""РАЗВЕДКА (hypothesis=explore, test не открыт). Отбор val+loto (решение Павла 2026-10-31) для linmap, linmap_prompt, linmap_mix; Qwen3-1.7B, те же 110 промптов train; свежий набор переноса seed 505 (знакомые типы: 33 меняющих и 17 инвариантных пар; новые: 36 и 14).

1) Что выбрал новый отбор. linmap — то же, что по val (L18, α1.5). linmap_prompt — L18/α2.0 вместо L24/α4.0 (лучшая по val конфигурация была на краю сетки, на отложенном типе 0.587 против 0.717). linmap_mix — L18/α1.5 вместо α1.0. Согласие val и отложенного типа по 30 конфигурациям слабое: Spearman 0.33 (linmap), −0.58 (linmap_prompt), 0.05 (linmap_mix) — они ранжируют настройки по-разному.

2) Перенос (успех / чувств. / специф.; эпизоды):
- знакомые типы: база 0.51/0.27/0.29 (0.46); LoRA 0.94/0.94/0.88 (0.73); linmap 0.75/0.64/0.53 (0.56); linmap_prompt 0.73/0.55/0.59 (0.60); linmap_mix 0.84/0.79/0.71 (0.68); выбор по одному val: linmap_prompt@val 0.70/0.45/0.59, linmap_mix@val 0.71/0.52/0.59.
- новые типы: база 0.45/0.11/0.29 (0.33); LoRA 0.89/0.89/0.71 (0.84); linmap 0.62/0.44/0.36 (0.40); linmap_prompt 0.78/0.75/0.43 (0.70); linmap_mix 0.76/0.69/0.43 (0.65); linmap_prompt@val 0.78/0.75/0.50, linmap_mix@val 0.68/0.56/0.36.

3) val+loto против одного val (парно по задачам переноса; посчитано отдельно по записанным решениям, crossslice.by_task + metrics.paired_bootstrap): linmap_mix +0.13 [+0.04, +0.22] на знакомых и +0.08 [+0.03, +0.13] на новых; linmap_prompt +0.03 [−0.06, +0.12] и +0.00 [−0.08, +0.08]; linmap — та же конфигурация. Новый отбор не ухудшил ни один метод и дал значимый выигрыш там, где val не различал конфигурации (у linmap_mix на val было шесть ничьих).

4) linmap_mix с отбором val+loto — первая карта без градиента, которая держит и знакомые, и новые типы: 0.84 и 0.76 (LoRA 0.94 и 0.89; linmap_mix − LoRA −0.10 [−0.21, +0.01] и −0.13 [−0.21, −0.06]); эпизоды 0.68 и 0.65 против 0.73 и 0.84. В 10 при отборе по val он был ≈ linmap. linmap_prompt на новых типах 0.78 — третий свежий набор подряд (0.83, 0.78, 0.78).

5) Специфичность на этом наборе низкая у всех, включая базу (0.29). На новых типах у поправок 0.43 против 0.71 у LoRA: инвариантные пары относительно базы они не ломают, но и не чинят так, как LoRA. Это главный оставшийся разрыв.

Ограничения: один свежий набор, 50 задач на часть, интервалы широкие; оба отбора сравниваются на одних задачах; всё — разведка.""",
    next_step="""1) Правило отбора для H4′ — val+loto для обоих срезов (записать в журнал до ноутбука 12). 2) Ноутбук 12: глобальный срез g1 — linmap и linmap_prompt против LoRA (H4′), linmap_mix разведочно. 3) Выигрыш linmap_mix с val+loto подтвердить на ещё одном свежем наборе или на test после заморозки. 4) Специфичность на новых типах — главный разрыв с LoRA; искать, чем его закрыть без градиента.""",
)))

## 2026-10-31 18:11 · прогон `20261031-160942_loto-select-qwen3-1.7b_bd79ce`

**Гипотеза:** explore · **Ноутбук:** notebooks/11_loto_select.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, transfer · **git:** `675e567` (есть незакоммиченные изменения)

**Зачем:** Отбор val+loto (отложенный тип требований внутри train) для linmap, linmap_prompt, linmap_mix.

**Отличия от конфига по умолчанию:** `act.alphas` = [1.0, 1.5, 2.0, 3.0, 4.0]; `act.ranks` = [8]; `act.select` = 'val+loto'; `data.n_tasks` = 160; `episode.on_val` = False; `eval.methods` = ['baseline', 'lora', 'linmap', 'linmap_prompt', 'linmap_mix']; `model.name` = 'Qwen/Qwen3-1.7B'; `transfer.episode_methods` = ['baseline', 'lora', 'linmap', 'linmap_prompt', 'linmap_mix']; `transfer.seed` = 505

**⚠ Флаги протокола:**
- незакоммиченные изменения в src/docs/notebooks/configs — прогон не воспроизводится по коммиту

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.621 | 0.526 | 0.400 | — |  | — | — | 831 | 10.000 | 400.600 |
| lora | 0.948 | 1.000 | 0.800 | 0.328 | [+0.190, +0.466] | 19.000 | 0.000 | 831 | 10.000 | 299.000 |
| linmap | 0.862 | 0.842 | 0.600 | 0.241 | [+0.121, +0.379] | 16.000 | 2.000 | 831 | 10.000 | 410.600 |
| linmap_prompt | 0.724 | 0.579 | 0.600 | 0.103 | [+0.000, +0.224] | 8.000 | 2.000 | 831 | 10.000 | 298.000 |
| linmap_mix | 0.793 | 0.737 | 0.600 | 0.172 | [+0.069, +0.293] | 11.000 | 1.000 | 831 | 10.000 | 413.100 |
| linmap_prompt@val | 0.776 | 0.684 | 0.600 | 0.155 | [+0.052, +0.276] | 11.000 | 2.000 | 831 | 10.000 | 410.900 |
| linmap_mix@val | 0.810 | 0.789 | 0.600 | 0.190 | [+0.086, +0.310] | 11.000 | 0.000 | 831 | 10.000 | 421.600 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.510 | 0.273 | 0.294 | — |  | — | — | 0.460 | — | 886 | 10.000 | 470.600 |
| lora | 0.940 | 0.939 | 0.882 | 0.430 | [+0.320, +0.540] | 44.000 | 1.000 | 0.730 | 0.270 | 886 | 10.000 | 496.900 |
| linmap | 0.750 | 0.636 | 0.529 | 0.240 | [+0.120, +0.360] | 30.000 | 6.000 | 0.560 | 0.100 | 886 | 10.000 | 546.400 |
| linmap_prompt | 0.730 | 0.545 | 0.588 | 0.220 | [+0.130, +0.320] | 25.000 | 3.000 | 0.600 | 0.140 | 886 | 10.000 | 624.700 |
| linmap_mix | 0.840 | 0.788 | 0.706 | 0.330 | [+0.220, +0.440] | 35.000 | 2.000 | 0.680 | 0.220 | 886 | 10.000 | 603.400 |
| linmap_prompt@val | 0.700 | 0.455 | 0.588 | 0.190 | [+0.070, +0.310] | 27.000 | 8.000 | — | — | 886 | 10.000 | 641.400 |
| linmap_mix@val | 0.710 | 0.515 | 0.588 | 0.200 | [+0.120, +0.290] | 20.000 | 0.000 | — | — | 886 | 10.000 | 696.200 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.450 | 0.111 | 0.286 | — |  | — | — | 0.330 | — | 853 | 10.000 | 401.700 |
| lora | 0.890 | 0.889 | 0.714 | 0.440 | [+0.340, +0.540] | 45.000 | 1.000 | 0.840 | 0.510 | 853 | 10.000 | 418.100 |
| linmap | 0.620 | 0.444 | 0.357 | 0.170 | [+0.100, +0.240] | 17.000 | 0.000 | 0.400 | 0.070 | 853 | 10.000 | 408.300 |
| linmap_prompt | 0.780 | 0.750 | 0.429 | 0.330 | [+0.240, +0.420] | 33.000 | 0.000 | 0.700 | 0.370 | 853 | 10.000 | 409.100 |
| linmap_mix | 0.760 | 0.694 | 0.429 | 0.310 | [+0.230, +0.400] | 31.000 | 0.000 | 0.650 | 0.320 | 853 | 10.000 | 408.500 |
| linmap_prompt@val | 0.780 | 0.750 | 0.500 | 0.330 | [+0.240, +0.420] | 34.000 | 1.000 | — | — | 853 | 10.000 | 408.500 |
| linmap_mix@val | 0.680 | 0.556 | 0.357 | 0.230 | [+0.160, +0.310] | 23.000 | 0.000 | — | — | 853 | 10.000 | 408.400 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L16/post_message: AUROC 0.980, acc 0.882 при перестановке 0.573
- H2: ✔ support — различает (H1), но ошибается в 47% меняющих пар: чувствительность 0.526, специфичность 0.400
- H3: ? inconclusive (val) — на val: успех 0.621 → 0.655 при L16/pre_action/α=1.0; решает только test
- H4: · not_run — нужны steering и LoRA, выбранные на val и оценённые на одних задачах
- ALT: · not_run — steering не сравнивался с промптом / пересказом / явным состоянием

**Вывод:** РАЗВЕДКА (hypothesis=explore, test не открыт). Отбор val+loto (решение Павла 2026-10-31) для linmap, linmap_prompt, linmap_mix; Qwen3-1.7B, те же 110 промптов train; свежий набор переноса seed 505 (знакомые типы: 33 меняющих и 17 инвариантных пар; новые: 36 и 14).

1) Что выбрал новый отбор. linmap — то же, что по val (L18, α1.5). linmap_prompt — L18/α2.0 вместо L24/α4.0 (лучшая по val конфигурация была на краю сетки, на отложенном типе 0.587 против 0.717). linmap_mix — L18/α1.5 вместо α1.0. Согласие val и отложенного типа по 30 конфигурациям слабое: Spearman 0.33 (linmap), −0.58 (linmap_prompt), 0.05 (linmap_mix) — они ранжируют настройки по-разному.

2) Перенос (успех / чувств. / специф.; эпизоды):
- знакомые типы: база 0.51/0.27/0.29 (0.46); LoRA 0.94/0.94/0.88 (0.73); linmap 0.75/0.64/0.53 (0.56); linmap_prompt 0.73/0.55/0.59 (0.60); linmap_mix 0.84/0.79/0.71 (0.68); выбор по одному val: linmap_prompt@val 0.70/0.45/0.59, linmap_mix@val 0.71/0.52/0.59.
- новые типы: база 0.45/0.11/0.29 (0.33); LoRA 0.89/0.89/0.71 (0.84); linmap 0.62/0.44/0.36 (0.40); linmap_prompt 0.78/0.75/0.43 (0.70); linmap_mix 0.76/0.69/0.43 (0.65); linmap_prompt@val 0.78/0.75/0.50, linmap_mix@val 0.68/0.56/0.36.

3) val+loto против одного val (парно по задачам переноса; посчитано отдельно по записанным решениям, crossslice.by_task + metrics.paired_bootstrap): linmap_mix +0.13 [+0.04, +0.22] на знакомых и +0.08 [+0.03, +0.13] на новых; linmap_prompt +0.03 [−0.06, +0.12] и +0.00 [−0.08, +0.08]; linmap — та же конфигурация. Новый отбор не ухудшил ни один метод и дал значимый выигрыш там, где val не различал конфигурации (у linmap_mix на val было шесть ничьих).

4) linmap_mix с отбором val+loto — первая карта без градиента, которая держит и знакомые, и новые типы: 0.84 и 0.76 (LoRA 0.94 и 0.89; linmap_mix − LoRA −0.10 [−0.21, +0.01] и −0.13 [−0.21, −0.06]); эпизоды 0.68 и 0.65 против 0.73 и 0.84. В 10 при отборе по val он был ≈ linmap. linmap_prompt на новых типах 0.78 — третий свежий набор подряд (0.83, 0.78, 0.78).

5) Специфичность на этом наборе низкая у всех, включая базу (0.29). На новых типах у поправок 0.43 против 0.71 у LoRA: инвариантные пары относительно базы они не ломают, но и не чинят так, как LoRA. Это главный оставшийся разрыв.

Ограничения: один свежий набор, 50 задач на часть, интервалы широкие; оба отбора сравниваются на одних задачах; всё — разведка.

**Следующий шаг:** 1) Правило отбора для H4′ — val+loto для обоих срезов (записать в журнал до ноутбука 12). 2) Ноутбук 12: глобальный срез g1 — linmap и linmap_prompt против LoRA (H4′), linmap_mix разведочно. 3) Выигрыш linmap_mix с val+loto подтвердить на ещё одном свежем наборе или на test после заморозки. 4) Специфичность на новых типах — главный разрыв с LoRA; искать, чем его закрыть без градиента.
